In [13]:
import pandas as pd

PATH = "/lustre/fs5/jarv_lab/scratch/adenisova/Inno_2025/tmp/"
file_name = "TOGA_ALL_new_tree_reest.txt"

rer_tree = f"{PATH}/{file_name}"

df = pd.read_csv(
    rer_tree, 
    sep = "\s+",
    names = ["gene", "tree"]
)

df.dropna(inplace=True)
df

df.to_csv(
    f"{PATH}/checked_{file_name}",
    sep = "\t",
    index = False,
    header = False
)

In [9]:
from ete3 import Tree

def canonical_topology(tree, taxa):
    """
    Приводим дерево к каноническому виду:
    - обрезаем по taxa
    - убираем длины ветвей
    - сортируем потомков
    """
    t = tree.copy()
    t.prune(taxa, preserve_branch_length=False)

    for node in t.traverse():
        node.dist = 0.0

    return t.write(format=9)  # topology-only, sorted


trees = {}
with open(rer_tree) as f:
    for line in f:
        gene, newick = line.strip().split("\t")
        trees[gene] = Tree(newick, format=1)

# выбираем эталон
ref_gene = list(trees.keys())[0]
ref_tree = trees[ref_gene]

# общий набор таксонов
common_taxa = set(ref_tree.get_leaf_names())
for t in trees.values():
    common_taxa &= set(t.get_leaf_names())

ref_topo = canonical_topology(ref_tree, common_taxa)

matching = []
discordant = []

for gene, tree in trees.items():
    try:
        topo = canonical_topology(tree, common_taxa)
        if topo == ref_topo:
            matching.append(gene)
        else:
            discordant.append(gene)
    except Exception as e:
        discordant.append(gene)

print("Согласуются:", matching)
print("Не согласуются:", discordant)

Согласуются: ['A1CF_rna-XM_421561.6', 'AACS_ENSGALT00000004577.6', 'AADAT_rna-XM_004940922.3', 'AAK1_rna-XM_015297334.2', 'AANAT_rna-XM_015295119.2', 'AARSD1_ENSGALT00000004541.7', 'AASDH_ENSGALT00000022357.6', 'AASS_rna-XM_015282251.2', 'AMH_rna-NM_205030.1', 'AMIGO3_rna-XM_025154610.1', 'AMN1_rna-XM_416368.5', 'AMOTL1_rna-XM_425649.6', 'AMOT_rna-XM_015278595.2', 'ANKS1B_rna-XM_015284697.2', 'ANKS4B_rna-XM_424609.6', 'ASB6_ENSGALT00000006983.5', 'ASB9_rna-XM_015273223.2', 'ATP11A_rna-XM_015277887.2', 'ATP11C_ENSGALT00000105191.1', 'ATP13A3_rna-XM_004943327.3', 'ATP1A1_ENSGALT00000047913.2', 'ATP1B3_ENSGALT00000107916.1', 'ATP23_rna-XM_015281404.2', 'BTBD11_ENSGALT00000082819.2', 'BTBD19_rna-XM_025153075.1', 'BTBD2_rna-XM_015299872.2', 'BTBD6_rna-XM_004936422.3', 'BTBD8_rna-XM_015290685.2', 'BTC_rna-XM_015276318.2', 'BTF3L4_ENSGALT00000017200.5', 'BTG4_rna-XM_015298002.2', 'BTRC_ENSGALT00000048079.2', 'BUB1_rna-NM_001012870.1', 'BUD13_rna-XM_015298026.2', 'BUD31_rna-XM_414798.6', 'CACY

In [11]:
print(ref_tree)


   /-Smithornis_capensis
  |
  |--Pitta_novaeguineae
  |
  |      /-Tyrannus_savana
  |   /-|
  |  |   \-Manacus_vitellinus
  |  |
  |  |                        /-Alauda_arvensis
  |  |                       |
  |  |                       |            /-Phylloscopus_sibilatrix
  |  |                       |         /-|
  |  |                       |        |  |   /-Cettia_cetti
  |  |                     /-|        |   \-|
  |  |                    |  |      /-|      \-Aegithalos_caudatus
  |  |                    |  |     |  |
  |  |                    |  |     |  |   /-Sylvia_atricapilla
  |  |                    |  |   /-|   \-|
  |  |                    |  |  |  |      \-Leiothrix_lutea
  |  |                  /-|   \-|  |
  |  |                 |  |     |   \-Hirundo_rustica
  |  |                 |  |     |
  |  |                 |  |      \-Hippolais_icterina
  |  |                 |  |
  |  |                 |  |      /-Cyanistes_caeruleus
  |  |                 |  |   /-|
--|

In [ ]:
df.to_csv(
    f"{PATH}/checked_{file_name}",
    sep = "\t",
    index = False,
    header = False
)

df_genes = pd.read_csv(
    f"{PATH}/TOGA_ALL_polished_phy_stat.tsv",
    sep = "\t",
    names = ["gene","len_seq","number_of_reqs","sp_names"]
)
# df_genes = df_genes[df_genes["number_of_reqs"] > 200]

In [4]:
df.merge(df_genes, how = "inner")[["gene", "tree"]].to_csv(
    f"{PATH}/checked_{file_name}",
    sep = "\t",
    index = False,
    header = False
)

In [39]:
from ete3 import Tree

t = Tree(df[df["gene"] == "KCNE4_rna-XM_025153654.1"]["tree"].item(), format = 1)
all_leafs = set(t.get_leaf_names())

df["uniq"] = df["tree"].apply(lambda x: set(Tree(x, format = 1).get_leaf_names()) - all_leafs)

In [42]:
df[df["uniq"].apply(lambda x: len(x) > 0)]

,gene,tree,uniq



KeyboardInterrupt



In [2]:
df[df["gene"] == "AACS_ENSGALT00000004577.6"]["tree"].item()

'(Tyrannus_savana:0.0491120196,Oxyruncus_cristatus:0.0325447824,((((Pipra_filicauda:0.0075016437,Manacus_vitellinus:0.0117625149)mrcaott105307ott128718:0.0058520004,Lepidothrix_velutina:0.0051029829)mrcaott49248ott105307:0.0099837890,Chiroxiphia_lanceolata:0.0182827814)mrcaott25827ott49248:0.0103075759,(Grallaria_varia:0.0631984475,((Pitta_sordida:0.0540914173,Smithornis_capensis:0.0475988127)mrcaott33874ott50165:0.0221558033,((((((((((((((Zosterops_pallidus:0.0032307347,Zosterops_japonicus:0.0116292191)mrcaott22609ott66394:0.0386805161,(Sylvia_atricapilla:0.0231012889,Sylvia_borin:0.0154195268)mrcaott11419ott830714:0.0497882598)mrcaott1488ott7487:0.0249450688,Pycnonotus_jocosus:0.0527473233)mrcaott1488ott4470:0.0193057750,(Phylloscopus_sibilatrix:0.0560699706,(Cettia_cetti:0.0238810175,Aegithalos_caudatus:0.0579846961)mrcaott5805ott45508:0.0072648785)mrcaott5805ott45321:0.0073445121)mrcaott1488ott5805:0.0080079562,(Riparia_riparia:0.0177073875,Hirundo_rustica:0.0168837582)mrcaott16185

In [3]:
df

,gene,tree
0,A1CF_rna-XM_421561.6,"(Tyrannus_savana:0.1798496778,Oxyruncus_crista..."
1,A4GALT_rna-XM_015290446.2,"(Tyrannus_savana:0.2021741914,(((Pipra_filicau..."
2,A4GNT_rna-XM_426692.5,"(Tyrannus_savana:0.0746472526,Oxyruncus_crista..."
3,AAAS_ENSGALT00000088467.2,"(Pipra_filicauda:0.0079899640,Lepidothrix_velu..."
4,AACS_ENSGALT00000004577.6,"(Tyrannus_savana:0.0491120196,Oxyruncus_crista..."
...,...,...
3718,ZFPM2_rna-XM_418380.6,"(Tyrannus_savana:0.0267434121,Oxyruncus_crista..."
3719,ZFYVE19_ENSGALT00000013788.6,"(Tyrannus_savana:0.0570726543,Oxyruncus_crista..."
3720,ZFYVE1_rna-XM_015287087.2,"(Tyrannus_savana:0.0404787967,Oxyruncus_crista..."
3721,ZFYVE21_ENSGALT00000060862.2,"(Tyrannus_savana:0.0573798288,Oxyruncus_crista..."
